<a href="https://colab.research.google.com/github/AICHUCKY/Ai-with-Chucky-Colab-Notebooks/blob/main/MatAnyone2%20Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 🔴 **Brought to you by [AI With Chucky](https://youtube.com/@AIWithChucky)**
*Subscribe for more AI tutorials, workflows, and optimization tips!*

In [ ]:
# @title 🛠️ Phase 1: High-Speed Installation
import os, subprocess, sys

def stream_cmd(cmd):
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, shell=True, text=True)
    for line in process.stdout:
        sys.stdout.write(line)
        sys.stdout.flush()
    process.wait()

subprocess.run("apt-get install -y aria2 ffmpeg", shell=True)

if not os.path.exists("/content/MatAnyone2"):
    stream_cmd("git clone https://github.com/pq-yang/MatAnyone2.git /content/MatAnyone2")

os.chdir("/content/MatAnyone2")
stream_cmd("pip install -e . && pip install omegaconf decord timm gradio psutil imageio-ffmpeg ffmpeg-python git+https://github.com/facebookresearch/segment-anything.git")

os.makedirs("pretrained_models", exist_ok=True)
weights = {
    "matanyone2.pth": "https://github.com/pq-yang/MatAnyone2/releases/download/v1.0.0/matanyone2.pth",
    "sam_vit_h_4b8939.pth": "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth"
}
for name, url in weights.items():
    stream_cmd(f"aria2c -c -x 16 -s 16 -k 1M {url} -d pretrained_models -o {name}")

In [ ]:
# @title 🎨 Phase 2: Launch MatAnyone Studio Pro
import os
import subprocess

repo_path = "/content/MatAnyone2/hugging_face"

if not os.path.exists(repo_path):
    print("\u274c Please run Phase 1 first.")
else:
    os.chdir(repo_path)
    subprocess.run(["git", "checkout", "app.py"], capture_output=True)

    with open("app.py", "r", encoding="utf-8") as f:
        code = f.read()

    code = code.replace("gr.Markdown(description)", "gr.Markdown(description, visible=False)")
    code = code.replace("gr.Markdown(article)", "gr.Markdown(article, visible=False)")
    code = code.replace(
        'with gr.Accordion("\ud83d\udcd5 Video Tutorial (click to expand)", open=False, elem_classes="custom-bg"):',
        'with gr.Accordion("\ud83d\udcd5 Video Tutorial", open=False, visible=False):'
    )
    code = code.replace("#000000, #2dc464", "#ff8c00, #ffffff")
    code = code.replace("theme=gr.themes.Monochrome()", 'theme=gr.themes.Soft(primary_hue="orange", neutral_hue="slate")')

    safe_css = """
    .gradio-container { background-color: #0d1117 !important; color: #e1e7ef !important; }
    .gr-button-primary, .green_button { background: linear-gradient(90deg, #ff8c00 0%, #ff4500 100%) !important; border: none !important; font-weight: bold !important; color: white !important; border-radius: 8px !important; }
    .gr-button-secondary, .new_button { background-color: #21262d !important; border: 1px solid #30363d !important; color: white !important; }
    footer, .gr-examples { display: none !important; }
    """
    code = code.replace('my_custom_css = """', f'my_custom_css = """\n{safe_css}')
    code = code.replace("demo.launch(share=True)", "demo.launch(share=True, debug=True)")

    with open("app.py", "w", encoding="utf-8") as f:
        f.write(code)

    get_ipython().system('PYTHONUNBUFFERED=1 python app.py')